# Advanced: Multi-Strategy Field Splitting with PAMOLA.CORE

**Goal**: Master all 3 field splitting strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Logical Grouping - Organize by semantic categories (personal, work, contact)
- **Strategy 2**: Data Type Grouping - Split by field types (numeric, categorical, temporal)
- **Strategy 3**: Sensitivity-Based - Group by data sensitivity levels (public, internal, confidential)
- Compare field organization across strategies
- Analyze schema complexity and field distribution
- Production deployment patterns for data governance

**Prerequisites:**
- Completed the simple notebook
- Understanding of data organization concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/
        └── 02_split_fields_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display

print("🔍 Detecting PAMOLA installation...\n")

# Auto-detect project root (search up 6 levels for pamola_core/)
current_dir = Path.cwd()
project_root = current_dir
pamola_found = False

for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"✓ Found pamola_core at level {level}: {project_root}")
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break
    project_root = project_root.parent

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.splitting.split_fields_op import SplitFieldsOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 18 columns)
- Sample records (first 10 rows)
- Column statistics (types, ranges, unique counts)

**Dataset features:**
- 1000 employee records
- 18 diverse fields (ID, personal, work, contact, temporal, financial)
- Mixed data types (strings, numbers, dates, booleans)
- Varied sensitivity levels for testing strategies

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'split_fields_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic employee data
    first_names = ['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 'Linda', 
                   'William', 'Barbara', 'David', 'Elizabeth', 'Richard', 'Susan', 'Joseph', 'Jessica']
    last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
                  'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson', 'Thomas']
    
    departments = ['IT', 'HR', 'Sales', 'Finance', 'Marketing', 'Operations', 'Legal', 'R&D']
    job_titles = ['Manager', 'Senior Manager', 'Director', 'VP', 'Analyst', 'Senior Analyst', 
                  'Specialist', 'Coordinator', 'Associate', 'Lead']
    cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia', 
              'San Antonio', 'San Diego', 'Dallas', 'Austin']
    countries = ['USA', 'Canada', 'UK', 'Germany', 'France']
    
    df = pd.DataFrame({
        # Identifier
        'employee_id': [f'EMP{str(i).zfill(5)}' for i in range(1, n_records + 1)],
        
        # Personal Information
        'first_name': np.random.choice(first_names, n_records),
        'last_name': np.random.choice(last_names, n_records),
        'birth_date': pd.date_range('1960-01-01', periods=n_records, freq='10D'),
        'gender': np.random.choice(['Male', 'Female', 'Other'], n_records, p=[0.48, 0.48, 0.04]),
        'marital_status': np.random.choice(['Single', 'Married', 'Divorced', 'Widowed'], n_records),
        'nationality': np.random.choice(countries, n_records),
        
        # Work Information
        'department': np.random.choice(departments, n_records),
        'job_title': np.random.choice(job_titles, n_records),
        'hire_date': pd.date_range('2010-01-01', periods=n_records, freq='5D'),
        'employment_type': np.random.choice(['Full-time', 'Part-time', 'Contract'], n_records, p=[0.7, 0.2, 0.1]),
        'years_experience': np.random.randint(0, 30, n_records),
        
        # Financial Information
        'salary': np.random.randint(40000, 200000, n_records),
        'bonus': np.random.randint(0, 50000, n_records),
        'stock_options': np.random.choice([True, False], n_records, p=[0.3, 0.7]),
        
        # Contact Information
        'email': [f'emp{i}@company.com' for i in range(1, n_records + 1)],
        'phone': [f'+1-555-{np.random.randint(100, 999)}-{np.random.randint(1000, 9999)}' for _ in range(n_records)],
        'city': np.random.choice(cities, n_records)
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records (first 10):")
print(df.head(10))

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_bool_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): boolean")
    else:
        non_null = df[col].dropna()
        if len(non_null) > 0:
            print(f"  {col:20s} ({dtype_str:10s}): min={non_null.min()}, max={non_null.max()}")

print(f"\n💡 Perfect dataset for testing all 3 field splitting strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit id_field for all strategies
   - `"id_field": "employee_id"` - Change to YOUR dataset's ID column
2. Run to validate field and setup environment

**What you'll see:**
- Field validation (✓ id_field exists)
- Total field count in dataset
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Configuration:**
```python
FIELD_CONFIG = {
    "id_field": "employee_id"  # CUSTOMIZE: Your ID column
}
```

**Note:** ID field must exist in dataset. All strategies will include this field in outputs

In [ ]:
# Field configuration - CUSTOMIZE THIS!
FIELD_CONFIG = {
    "id_field": "employee_id"  # Field used as identifier in all strategies
}

# Validate field exists in dataset
print("📋 Validating field configuration...\n")
id_field = FIELD_CONFIG["id_field"]

if id_field not in df.columns:
    raise ValueError(
        f"❌ Field '{id_field}' not found!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update FIELD_CONFIG"
    )

print(f"  ✓ ID field: '{id_field}'")
print(f"    Will be included in all field groups\n")
print(f"  ✓ Total fields in dataset: {len(df.columns)}")
print(f"    Available for grouping: {len(df.columns) - 1} (excluding ID)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy_fields",
    description="Multi-strategy field splitting",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Logical Grouping

**How to use:**
- Review pre-configured semantic groups
- Run to execute logical field grouping
- Monitor progress bar (6 steps)

**Key parameters:**
- `field_groups`: Dictionary defining logical categories
  - `personal_info`: Name, birth date, gender, marital status, nationality (7 fields)
  - `work_info`: Department, job title, hire date, employment type, experience (5 fields)
  - `financial_info`: Salary, bonus, stock options (3 fields)
  - `contact_info`: Email, phone, city (3 fields)
- `include_id_field=True`: Add employee_id to each group

**What you'll see:**
- Configuration summary with 4 logical groups
- Progress: validation → grouping → saving → metrics → viz
- Completion time (1-2 seconds)
- Output: 4 files (one per logical category)

**Validate:**
- ✅ Execution time <5 seconds
- ✅ 4 output files created
- ✅ Each file contains ID + category fields
- ✅ Status = "completed"

**Note:** Best for semantic organization and domain-driven data management

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: LOGICAL GROUPING")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Logical",
    unit="steps",
    track_memory=True,
    level=0
)

# Define logical field groups
field_groups_s1 = {
    "personal_info": ['first_name', 'last_name', 'birth_date', 'gender', 
                      'marital_status', 'nationality'],
    "work_info": ['department', 'job_title', 'hire_date', 'employment_type', 
                  'years_experience'],
    "financial_info": ['salary', 'bonus', 'stock_options'],
    "contact_info": ['email', 'phone', 'city']
}

# Create operation
operation_s1 = SplitFieldsOperation(
    # Core parameters
    id_field=id_field,
    field_groups=field_groups_s1,
    include_id_field=True,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Field groups: {len(field_groups_s1)}")
for group_name, fields in field_groups_s1.items():
    print(f"    {group_name}: {len(fields)} fields")

print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_logical',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Count output files
output_dir_s1 = task_dir / 'strategy1_logical' / 'output'
if output_dir_s1.exists():
    output_files_s1 = list(output_dir_s1.glob('*.csv'))
    print(f"\n📊 Generated {len(output_files_s1)} output files")
    
    # Show field counts
    for f in sorted(output_files_s1, key=lambda x: x.name):
        group_df = pd.read_csv(f)
        parts = f.stem.split('_')
        try:
            output_idx = parts.index('output')
            group_name = '_'.join(parts[4:output_idx])
        except (ValueError, IndexError):
            group_name = f.stem
        print(f"  • {group_name}: {len(group_df.columns)} fields, {len(group_df)} records")

## STRATEGY 2: Data Type Grouping

**How to use:**
- Automatically groups fields by data type
- No semantic knowledge needed
- Run to execute type-based grouping

**Key parameters:**
- `field_groups`: Dictionary defining type-based categories
  - `numeric_fields`: All integer/float columns (4 fields)
  - `categorical_fields`: String/categorical columns (10 fields)
  - `temporal_fields`: Date/datetime columns (2 fields)
  - `boolean_fields`: Boolean columns (1 field)

**What you'll see:**
- Configuration confirmation
- Progress: validation → type detection → grouping → metrics → viz
- Completion time (1-2 seconds)
- Output: 4 files (one per data type category)

**Validate:**
- ✅ Execution time <5 seconds
- ✅ 4 output files created
- ✅ Consistent data types within each group
- ✅ Status = "completed"

**Note:** Best for technical data organization and storage optimization

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: DATA TYPE GROUPING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Data Types",
    unit="steps",
    track_memory=True,
    level=0
)

# Define data type-based groups
field_groups_s2 = {
    "numeric_fields": ['years_experience', 'salary', 'bonus'],
    "categorical_fields": ['first_name', 'last_name', 'gender', 'marital_status', 
                           'nationality', 'department', 'job_title', 'employment_type',
                           'email', 'phone', 'city'],
    "temporal_fields": ['birth_date', 'hire_date'],
    "boolean_fields": ['stock_options']
}

operation_s2 = SplitFieldsOperation(
    # Core parameters
    id_field=id_field,
    field_groups=field_groups_s2,
    include_id_field=True,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Field groups: {len(field_groups_s2)}")
for group_name, fields in field_groups_s2.items():
    print(f"    {group_name}: {len(fields)} fields")

print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_datatypes',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Count output files
output_dir_s2 = task_dir / 'strategy2_datatypes' / 'output'
if output_dir_s2.exists():
    output_files_s2 = list(output_dir_s2.glob('*.csv'))
    print(f"\n📊 Generated {len(output_files_s2)} output files")
    
    # Show field counts
    for f in sorted(output_files_s2, key=lambda x: x.name):
        group_df = pd.read_csv(f)
        parts = f.stem.split('_')
        try:
            output_idx = parts.index('output')
            group_name = '_'.join(parts[4:output_idx])
        except (ValueError, IndexError):
            group_name = f.stem
        print(f"  • {group_name}: {len(group_df.columns)} fields, {len(group_df)} records")

## STRATEGY 3: Sensitivity-Based Grouping

**How to use:**
- Groups fields by data sensitivity level
- Supports data governance and compliance
- Run to execute sensitivity-based grouping

**Key parameters:**
- `field_groups`: Dictionary defining sensitivity levels
  - `public_data`: Non-sensitive fields (5 fields)
  - `internal_data`: Business-sensitive fields (7 fields)
  - `confidential_data`: Highly sensitive PII/financial (5 fields)

**What you'll see:**
- Configuration confirmation
- Progress: validation → classification → grouping → metrics → viz
- Completion time (1-2 seconds)
- Output: 3 files (one per sensitivity level)

**Validate:**
- ✅ Execution time <5 seconds
- ✅ 3 output files created
- ✅ Clear separation by sensitivity
- ✅ Status = "completed"

**Note:** Best for compliance, privacy protection, and access control management

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: SENSITIVITY-BASED GROUPING")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Sensitivity",
    unit="steps",
    track_memory=True,
    level=0
)

# Define sensitivity-based groups
field_groups_s3 = {
    "public_data": ['department', 'job_title', 'city', 'employment_type', 'years_experience'],
    "internal_data": ['first_name', 'last_name', 'hire_date', 'email', 'phone', 'nationality', 'stock_options'],
    "confidential_data": ['birth_date', 'gender', 'marital_status', 'salary', 'bonus']
}

operation_s3 = SplitFieldsOperation(
    # Core parameters
    id_field=id_field,
    field_groups=field_groups_s3,
    include_id_field=True,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Field groups: {len(field_groups_s3)}")
for group_name, fields in field_groups_s3.items():
    print(f"    {group_name}: {len(fields)} fields")

print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_sensitivity',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Count output files
output_dir_s3 = task_dir / 'strategy3_sensitivity' / 'output'
if output_dir_s3.exists():
    output_files_s3 = list(output_dir_s3.glob('*.csv'))
    print(f"\n📊 Generated {len(output_files_s3)} output files")
    
    # Show field counts
    for f in sorted(output_files_s3, key=lambda x: x.name):
        group_df = pd.read_csv(f)
        parts = f.stem.split('_')
        try:
            output_idx = parts.index('output')
            group_name = '_'.join(parts[4:output_idx])
        except (ValueError, IndexError):
            group_name = f.stem
        print(f"  • {group_name}: {len(group_df.columns)} fields, {len(group_df)} records")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all 3 strategies complete
- Review execution times and organization patterns
- Identify best strategy for your use case

**What you'll see:**
- Execution time comparison (fastest to slowest)
- Total processing time
- Group count comparison
- Field distribution analysis

**Strategy selection guide:**
- **Logical**: Semantic meaning, domain expertise
- **Data Type**: Technical optimization, storage efficiency
- **Sensitivity**: Privacy, compliance, access control

**Note:** Fastest strategy isn't always best - consider your organizational requirements

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Logical):      {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Data Types):   {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Sensitivity):  {elapsed_s3:6.2f}s")
print(f"  Total:                     {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

# Compare group counts
print("\n📈 Field Organization:")
print(f"  Strategy 1: {len(field_groups_s1)} groups (semantic categories)")
print(f"  Strategy 2: {len(field_groups_s2)} groups (data type categories)")
print(f"  Strategy 3: {len(field_groups_s3)} groups (sensitivity levels)")

# Field distribution
print("\n📋 Field Distribution:")
total_fields = len(df.columns) - 1  # Exclude ID
for strategy_name, field_groups in [('Logical', field_groups_s1), 
                                     ('Data Types', field_groups_s2),
                                     ('Sensitivity', field_groups_s3)]:
    avg_fields = sum(len(fields) for fields in field_groups.values()) / len(field_groups)
    print(f"  {strategy_name:15s}: {avg_fields:.1f} fields per group (avg)")

print("\n🎯 Key Insights:")
print(f"  • Total dataset: {len(df):,} records, {len(df.columns)} fields")
print(f"  • All strategies preserve data integrity")
print(f"  • Different organization patterns suit different needs")
print(f"  • ID field included in all output files")

## Step 5: Detailed Artifact Review

**How to use:**
- Review all generated artifacts from each strategy
- Auto-loads NEWEST files from each category
- Displays metrics and field lists inline

**What you'll see (per strategy):**
1. **Metrics JSON**: Split statistics, field counts, execution time
2. **Output Files**: Field groups with column lists
3. **Visualizations**: Schema charts (latest batch only)

**Validate:**
- ✅ All metrics show expected group counts
- ✅ Field lists match defined groups
- ✅ File sizes appropriate for content
- ✅ Visualizations show clear organization

**Note:** Only newest files shown. Multiple runs create versions - older artifacts excluded automatically

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_logical', 'Strategy 1: Logical Grouping'),
    ('strategy2_datatypes', 'Strategy 2: Data Type Grouping'),
    ('strategy3_sensitivity', 'Strategy 3: Sensitivity-Based')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data if isinstance(data, dict) else data.get('metrics', {})
                
                # Display key metrics
                print(f"   Input records: {metrics.get('total_input_records', 'N/A')}")
                print(f"   Input fields: {metrics.get('total_input_fields', 'N/A')}")
                print(f"   Number of groups: {metrics.get('number_of_splits', 'N/A')}")
                print(f"   Execution time: {metrics.get('execution_time_seconds', 0):.4f}s")
                
                # Show split info if available
                if 'split_info' in metrics:
                    print(f"\n   Group Details:")
                    for group_name, info in metrics['split_info'].items():
                        field_count = info.get('field_count', 0)
                        fields = info.get('included_fields', [])
                        print(f"     • {group_name}: {field_count} fields")
                        print(f"       → {', '.join(fields)}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Output Files (NEWEST BATCH)
    output_dir = strategy_dir / 'output'
    if output_dir.exists():
        output_files = sorted(
            list(output_dir.glob('*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if output_files:
            latest_time = output_files[0].stat().st_mtime
            latest_batch = [
                f for f in output_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📁 OUTPUT FILES: {len(latest_batch)} files")
            for f in sorted(latest_batch, key=lambda x: x.name):
                try:
                    group_df = pd.read_csv(f)
                    parts = f.stem.split('_')
                    try:
                        output_idx = parts.index('output')
                        group_name = '_'.join(parts[4:output_idx])
                    except (ValueError, IndexError):
                        group_name = f.stem
                    print(f"   • {group_name}: {len(group_df.columns)} fields, {len(group_df)} records")
                except Exception as e:
                    print(f"   ⚠️  Error reading {f.name}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports field groups from all strategies
- Organizes files by strategy for easy access

**What you'll see (per strategy):**
- File location paths
- Field count and record count per group
- Preview of sample records (first 3 rows)
- Column lists for each group

**Output structure:**
```
advanced_output/
├── strategy1_logical/
│   └── output/
│       ├── ...personal_info....csv
│       ├── ...work_info....csv
│       ├── ...financial_info....csv
│       └── ...contact_info....csv
├── strategy2_datatypes/
│   └── output/
│       ├── ...numeric_fields....csv
│       ├── ...categorical_fields....csv
│       ├── ...temporal_fields....csv
│       └── ...boolean_fields....csv
└── strategy3_sensitivity/
    └── output/
        ├── ...public_data....csv
        ├── ...internal_data....csv
        └── ...confidential_data....csv
```

**Note:** Files already saved during execution - this cell provides summary and verification

In [ ]:
print("=" * 80)
print("📦 EXPORT SUMMARY - ALL STRATEGIES")
print("=" * 80)

print(f"\n📂 Base directory: {task_dir}\n")

total_files = 0
total_groups = 0

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    output_dir = strategy_dir / 'output'
    
    if not output_dir.exists():
        continue
    
    print("=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    output_files = list(output_dir.glob('*.csv'))
    
    print(f"\n📁 Location: {output_dir}")
    print(f"📄 Files: {len(output_files)}\n")
    
    # Show each file
    for f in sorted(output_files, key=lambda x: x.name):
        try:
            group_df = pd.read_csv(f)
            
            # Extract group name
            parts = f.stem.split('_')
            try:
                output_idx = parts.index('output')
                group_name = '_'.join(parts[4:output_idx])
            except (ValueError, IndexError):
                group_name = f.stem
            
            print(f"  {group_name}")
            print(f"    Records: {len(group_df)}")
            print(f"    Fields: {len(group_df.columns)}")
            print(f"    Columns: {', '.join(group_df.columns)}")
            
            # Show first 3 rows
            print(f"\n    Sample (first 3):")
            print(group_df.head(3).to_string(index=False, max_colwidth=20))
            print()
            
        except Exception as e:
            print(f"  ⚠️  Error reading {f.name}: {e}")
    
    total_files += len(output_files)
    total_groups += len(output_files)

# ============================================================================
# GRAND SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📊 Grand Total:")
print(f"   Strategies executed: 3")
print(f"   Total files created: {total_files}")
print(f"   Total field groups: {total_groups}")
print(f"   Original dataset: {len(df)} records, {len(df.columns)} fields")

if 'elapsed_s1' in locals() and 'elapsed_s2' in locals() and 'elapsed_s3' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"\n⏱️  Total execution time: {total_time:.2f}s")
    print(f"   Average per strategy: {total_time/3:.2f}s")

print("\n" + "=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 3 field splitting strategies implemented and compared
- ✅ Multiple organization patterns demonstrated
- ✅ Schema analysis across strategies
- ✅ Production-ready artifacts generated

**Key takeaways:**
- **Logical Grouping** organizes by semantic meaning
- **Data Type Grouping** optimizes by technical characteristics
- **Sensitivity-Based** enforces privacy and compliance
- All strategies preserve data integrity
- ID field consistently included in all outputs

**Strategy recommendations:**
- **Use Logical** when: Domain knowledge important, semantic organization needed, user-facing applications
- **Use Data Type** when: Storage optimization required, technical processing needs, database design
- **Use Sensitivity** when: Privacy compliance mandatory, access control needed, regulatory requirements

**Data governance considerations:**
- Combine strategies for comprehensive organization
- Document field grouping rationale
- Implement access controls per sensitivity level
- Regular review and updates as data evolves

**Next steps:**
- Apply to your production datasets
- Define custom field groups
- Integrate with data governance framework
- Automate field classification

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)
- 📊 [Data Governance Guide](../../docs/governance.md)